# 01 · Conhecendo os dados de homicídios

Neste notebook, buscamos os dados do CDC, conferimos se há informações faltando e criamos gráficos para entender as diferenças entre os estados. É a etapa de **análise exploratória**, isto é, observar os dados antes de treinar um modelo.

In [ ]:
import sys
import pandas as pd
import sklearn

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Executável:", sys.executable)

In [ ]:
from pathlib import Path
from io import StringIO
import requests
import pandas as pd

url = "https://data.cdc.gov/resource/fpsi-y8tj.csv"

params = {
    "$where": "intent='All_Homicide' AND period IN ('2019','2020','2021','2022','2023','2024')",
    "$limit": 1000
}

resposta = requests.get(url, params=params, timeout=30)
resposta.raise_for_status()

homicidios = pd.read_csv(StringIO(resposta.text))

print("Total de registros:", len(homicidios))
display(homicidios.groupby("period").size())

In [ ]:
estados = homicidios[
    homicidios["name"] != "District of Columbia"
].copy()

print("Registros após filtro:", len(estados))
print("Duplicações:", estados.duplicated(["name", "period"]).sum())
print("Taxas ausentes:", estados["rate"].isna().sum())
print("Contagens ausentes:", estados["count_sup"].isna().sum())

display(estados.groupby("period")["name"].nunique())

In [ ]:
destino = Path("../data/raw/cdc_homicides_2019_2024.csv")

destino.parent.mkdir(parents=True, exist_ok=True)

homicidios.to_csv(destino, index=False)

print("Arquivo salvo em:", destino.resolve())

In [ ]:
from pathlib import Path
import pandas as pd

arquivo = Path("../data/raw/cdc_homicides_2019_2024.csv")

df = pd.read_csv(arquivo)

# Manter somente os 50 estados
df = df[df["name"] != "District of Columbia"].copy()

# Converter o ano para número inteiro
df["period"] = df["period"].astype(int)

display(df.head())
print("Dimensões:", df.shape)

In [ ]:
import matplotlib.pyplot as plt

media_anual = df.groupby("period")["rate"].mean()

display(media_anual)

media_anual.plot(
    kind="line",
    marker="o",
    figsize=(9, 5)
)

plt.title("Evolução das taxas de homicídios nos estados americanos")
plt.xlabel("Ano")
plt.ylabel("Média estadual por 100 mil habitantes")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x="period",
    y="rate"
)

plt.title("Distribuição das taxas de homicídios por estado")
plt.xlabel("Ano")
plt.ylabel("Homicídios por 100 mil habitantes")

plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
dados_2024 = df[df["period"] == 2024]

ranking = dados_2024.sort_values(
    by="rate",
    ascending=False
)

print("10 maiores taxas:")
display(ranking[["name", "rate"]].head(10))

print("10 menores taxas:")
display(ranking[["name", "rate"]].tail(10).sort_values("rate"))

## Interpretação descritiva

A análise das taxas de homicídios em 2024 mostrou diferenças expressivas entre os estados americanos. Mississippi apresentou a maior taxa (18,6), enquanto New Hampshire registrou a menor (1,3) por 100 mil habitantes.

Essas diferenças são descritivas e não permitem estabelecer uma relação causal com a legislação sobre armas de fogo. Para investigar a contribuição dessas leis à previsão dos homicídios, será necessário integrar informações legislativas e socioeconômicas.